In [38]:
import sys
import time
import random
import numpy as np
import pandas as pd
from functools import partial
import matplotlib.pyplot as plt
import pennylane as qp
import quimb
import pickle

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [18]:
def generate_bitstrings(array_length, bit_length, worstcase=False):
    if worstcase: return [f"{2**bit_length - 1:0{bit_length}b}" for _ in range(array_length)]
    return [f"{random.randint(0, 2**bit_length - 1):0{bit_length}b}" for _ in range(array_length)]


In [ ]:
@partial(qp.compile, basis_set="CNOT")
@qp.qnode(qp.device("default.qubit"))
def circuit(index, epsilon=0.05, seed=None):

    rng = np.random.default_rng(seed)
    qp.BasisState(index, wires=control_wires)
    qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)

    ops = qrom.decomposition()         # Decompose into primitive operations
    error_idx = rng.integers(len(ops)) # Choose a random location inside the QROM

    for i, op in enumerate(ops):
        qp.apply(op)
        if i == error_idx:

            error_wire = rng.choice(list(op.wires)) # Insert coherent error on one of the gates' wires
            qp.RZ(epsilon, wires=error_wire) # Small coherent calibration error

            # Alternative choices:
            # qp.RX(epsilon, wires=error_wire)
            # qp.RY(epsilon, wires=error_wire)

    return qp.state()

## Noise Injection

Depolarization Chennel: Probability of the combination of the occurrence of a bit flip, phase flip, and a bit-phase flip. Amplitude Damping and Phase Damping Channel, Bit Flip and Phase Flip Channels

https://s2.smu.edu/~mitch/ftp_dir/pubs/Mo25.pdf

# Coherent Phase Shift Errors

In [25]:
def evaluate_QROM(bitstrings, num_index_bits, num_swap_bits=0, bits_per_value=2, print_out=False):

    if(len(bitstrings) != 2**num_index_bits or len(bitstrings[0]) != bits_per_value):
        print(f"Error: Mismatch in size of bitstrings and expected number of num_index_bits and bits_per_value. ({len(bitstrings)}, {len(bitstrings[0])}) != ({2**num_index_bits}, {bits_per_value})")
        return -1, -1, -1, -1, -1
    
    control_wires = np.arange(num_index_bits)
    target_wires = np.arange(num_index_bits, num_index_bits + bits_per_value)
    work_wires = np.arange(target_wires[-1] + 1, target_wires[-1] + 1 + bits_per_value*(2**num_swap_bits - 1))

    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def noisy_circuit(index, epsilon=0.05, seed=None):
    
        rng = np.random.default_rng(seed)
        qp.BasisState(index, wires=control_wires)
        qrom = qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
    
        ops = qrom.decomposition()         # Decompose into primitive operations
        error_idx = rng.integers(len(ops)) # Choose a random location inside the QROM
    
        for i, op in enumerate(ops):
            qp.apply(op)
            if i == error_idx:
    
                error_wire = rng.choice(list(op.wires)) # Insert coherent error on one of the gates' wires
                qp.RZ(epsilon, wires=error_wire) # Small coherent calibration error
    
                # Alternative choices:
                # qp.RX(epsilon, wires=error_wire)
                # qp.RY(epsilon, wires=error_wire)
    
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()
    
    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def ideal_circuit(index):
        qp.BasisState(index, wires=control_wires)
        qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()

    r = qp.specs(ideal_circuit)(0)["resources"]

    num_qubits = len(control_wires) + len(target_wires) + len(work_wires)
    oneQ = r.gate_sizes.get(1,0)
    twoQ = r.gate_sizes.get(2,0)

    if print_out:
        print("Number of qubits: ", num_qubits)
        print("One-qubit gates: ", oneQ)
        print("Two-qubit gates: ", twoQ)
        print(f"Depth: {r.depth}")
        
    ideal_fidelities = []
    noisy_fidelities = []
    full_start_time = time.perf_counter()

    for index in range(2**num_index_bits):
        start_time = time.perf_counter()
        
        # Expected |address>|data> state
        probs_length = list(map(int, f"{index:0{num_index_bits}b}")) + list(map(int, bitstrings[index]))
        probs_index = int("".join(map(str, probs_length)), 2)

        # Ideal Circuit Probability of retrieving the correct address + data.
        ideal_probs, ideal_state = ideal_circuit(index)
        ideal_fidelity = ideal_probs[probs_index]
        ideal_fidelity = np.clip(np.real(ideal_fidelity), 0.0, 1.0)

        # Noisy Circuit Probability of retrieving the correct address + data.
        noisy_probs, noisy_state = noisy_circuit(index)
        noisy_fidelity = noisy_probs[probs_index]
        noisy_fidelity = np.clip(np.real(noisy_fidelity), 0.0, 1.0)

        elapsed = time.perf_counter() - start_time
        
        if print_out: 
            print(f"Index {index} took {elapsed:.6f} s")
            print(f"Ideal QROM Retrieval Fidelity for Index {index} is {ideal_fidelity}")
            print(f"Noisy QROM Retrieval Fidelity for Index {index} is {noisy_fidelity}")
        
        ideal_fidelities.append(float(ideal_fidelity))
        noisy_fidelities.append(float(noisy_fidelity))
        
    full_elapsed = time.perf_counter() - full_start_time    
    
    if print_out: 
        print(f"Final Ideal Retrieval Fidelity is {np.mean(ideal_fidelities)} and took {full_elapsed:.6f} s")
        print(f"Final Noisy Retrieval Fidelity is {np.mean(noisy_fidelities)} and took {full_elapsed:.6f} s")

Timing went from 5 seconds to 22 seconds

In [26]:
num_index_bits = 4
num_swap_bits = 2
bits_per_value = 4

bitstrings = generate_bitstrings(2**num_index_bits, bits_per_value, worstcase=True)
evaluate_QROM(bitstrings, num_index_bits, num_swap_bits, bits_per_value, print_out=True)


Number of qubits:  20
One-qubit gates:  939
Two-qubit gates:  480
Depth: 708
Index 0 took 21.666551 s
Ideal QROM Retrieval Fidelity for Index 0 is 0.9999999999999845
Noisy QROM Retrieval Fidelity for Index 0 is 0.9999999999999571
Index 0 took 21.666551 s
Index 1 took 21.159301 s
Ideal QROM Retrieval Fidelity for Index 1 is 0.999999999999984
Noisy QROM Retrieval Fidelity for Index 1 is 0.9999999999999611
Index 1 took 21.159301 s
Index 2 took 20.837641 s
Ideal QROM Retrieval Fidelity for Index 2 is 0.9999999999999831
Noisy QROM Retrieval Fidelity for Index 2 is 0.9999999999999529
Index 2 took 20.837641 s
Index 3 took 20.754224 s
Ideal QROM Retrieval Fidelity for Index 3 is 0.9999999999999836
Noisy QROM Retrieval Fidelity for Index 3 is 0.9999999999999586
Index 3 took 20.754224 s
Index 4 took 20.675542 s
Ideal QROM Retrieval Fidelity for Index 4 is 0.9999999999999887
Noisy QROM Retrieval Fidelity for Index 4 is 0.9999999999999621
Index 4 took 20.675542 s
Index 5 took 20.816886 s
Ideal QRO

# Deterministic Bit Flip

In [30]:
def evaluate_QROM(bitstrings, num_index_bits, num_swap_bits=0, bits_per_value=2, print_out=False):

    if(len(bitstrings) != 2**num_index_bits or len(bitstrings[0]) != bits_per_value):
        print(f"Error: Mismatch in size of bitstrings and expected number of num_index_bits and bits_per_value. ({len(bitstrings)}, {len(bitstrings[0])}) != ({2**num_index_bits}, {bits_per_value})")
        return -1, -1, -1, -1, -1
    
    control_wires = np.arange(num_index_bits)
    target_wires = np.arange(num_index_bits, num_index_bits + bits_per_value)
    work_wires = np.arange(target_wires[-1] + 1, target_wires[-1] + 1 + bits_per_value*(2**num_swap_bits - 1))

    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def noisy_circuit(index, epsilon=0.05, seed=None):
    
        rng = np.random.default_rng(seed)
        qp.BasisState(index, wires=control_wires)
        qrom = qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
    
        ops = qrom.decomposition()         # Decompose into primitive operations
        error_idx = rng.integers(len(ops)) # Choose a random location inside the QROM
    
        for i, op in enumerate(ops):
            qp.apply(op)
            if i == error_idx:
                error_wire = rng.choice(list(op.wires)) # Insert coherent error on one of the gates' wires
                qp.PauliX(wires=error_wire)
    
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()
    
    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def ideal_circuit(index):
        qp.BasisState(index, wires=control_wires)
        qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()

    r = qp.specs(ideal_circuit)(0)["resources"]

    num_qubits = len(control_wires) + len(target_wires) + len(work_wires)
    oneQ = r.gate_sizes.get(1,0)
    twoQ = r.gate_sizes.get(2,0)

    if print_out:
        print("Number of qubits: ", num_qubits)
        print("One-qubit gates: ", oneQ)
        print("Two-qubit gates: ", twoQ)
        print(f"Depth: {r.depth}")
        
    ideal_fidelities = []
    noisy_fidelities = []
    state_fidelities = []
    full_start_time = time.perf_counter()

    for index in range(2**num_index_bits):
        start_time = time.perf_counter()
        
        # Expected |address>|data> state
        probs_length = list(map(int, f"{index:0{num_index_bits}b}")) + list(map(int, bitstrings[index]))
        probs_index = int("".join(map(str, probs_length)), 2)

        # Ideal Circuit Probability of retrieving the correct address + data.
        ideal_probs, ideal_state = ideal_circuit(index)
        ideal_fidelity = ideal_probs[probs_index]
        ideal_fidelity = np.clip(np.real(ideal_fidelity), 0.0, 1.0)

        # Noisy Circuit Probability of retrieving the correct address + data.
        noisy_probs, noisy_state = noisy_circuit(index)
        noisy_fidelity = noisy_probs[probs_index]
        noisy_fidelity = np.clip(np.real(noisy_fidelity), 0.0, 1.0)

        # State Fidelity
        state_fidelity = np.abs(np.vdot(ideal_state, noisy_state))**2

        elapsed = time.perf_counter() - start_time
        
        if print_out: 
            print(f"Index {index} took {elapsed:.6f} s")
            print(f"Ideal QROM Retrieval Fidelity: {ideal_fidelity}")
            print(f"Noisy QROM Retrieval Fidelity: {noisy_fidelity}")
            print(f"State Fidelity: {state_fidelity}\n")
        
        ideal_fidelities.append(float(ideal_fidelity))
        noisy_fidelities.append(float(noisy_fidelity))
        state_fidelities.append(float(state_fidelity))
        
    full_elapsed = time.perf_counter() - full_start_time    
    
    if print_out: 
        print(f"Total Time: {full_elapsed:.6f} s")
        print(f"Final Ideal Retrieval Fidelity is {np.mean(ideal_fidelities)}")
        print(f"Final Noisy Retrieval Fidelity is {np.mean(noisy_fidelities)}")
        print(f"Final State Fidelity is {np.mean(state_fidelities)}")

In [31]:
num_index_bits = 4
num_swap_bits = 2
bits_per_value = 4

bitstrings = generate_bitstrings(2**num_index_bits, bits_per_value, worstcase=True)
evaluate_QROM(bitstrings, num_index_bits, num_swap_bits, bits_per_value, print_out=True)


Number of qubits:  20
One-qubit gates:  939
Two-qubit gates:  480
Depth: 708
Index 0 took 21.297586 s
Ideal QROM Retrieval Fidelity: 0.9999999999999845
Noisy QROM Retrieval Fidelity: 0.9999999999999565
State Fidelity: 4.946698254361968e-30

Index 1 took 21.772006 s
Ideal QROM Retrieval Fidelity: 0.999999999999984
Noisy QROM Retrieval Fidelity: 5.041251807673385e-30
State Fidelity: 2.6120410098326225e-30

Index 2 took 22.966374 s
Ideal QROM Retrieval Fidelity: 0.9999999999999831
Noisy QROM Retrieval Fidelity: 0.9999999999999534
State Fidelity: 3.708679273705013e-30

Index 3 took 22.437283 s
Ideal QROM Retrieval Fidelity: 0.9999999999999836
Noisy QROM Retrieval Fidelity: 0.9999999999999587
State Fidelity: 2.1142422948239294e-30

Index 4 took 23.452475 s
Ideal QROM Retrieval Fidelity: 0.9999999999999887
Noisy QROM Retrieval Fidelity: 0.9999999999999623
State Fidelity: 2.4322665437800255e-30

Index 5 took 22.778933 s
Ideal QROM Retrieval Fidelity: 0.999999999999988
Noisy QROM Retrieval Fid

### Let's see if we can backtrack

In [50]:
def evaluate_QROM(bitstrings, num_index_bits, num_swap_bits=0, bits_per_value=2, print_out=False):

    if(len(bitstrings) != 2**num_index_bits or len(bitstrings[0]) != bits_per_value):
        print(f"Error: Mismatch in size of bitstrings and expected number of num_index_bits and bits_per_value. ({len(bitstrings)}, {len(bitstrings[0])}) != ({2**num_index_bits}, {bits_per_value})")
        return -1, -1, -1, -1, -1
    
    control_wires = np.arange(num_index_bits)
    target_wires = np.arange(num_index_bits, num_index_bits + bits_per_value)
    work_wires = np.arange(target_wires[-1] + 1, target_wires[-1] + 1 + bits_per_value*(2**num_swap_bits - 1))

    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def noisy_circuit(index, epsilon=0.05, seed=None):
    
        rng = np.random.default_rng(seed)
        qp.BasisState(index, wires=control_wires)
        qrom = qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
    
        ops = qrom.decomposition()         # Decompose into primitive operations
        # error_idx = 0
        error_idx = rng.integers(len(ops)) # Choose a random location inside the QROM
    
        for i, op in enumerate(ops):
            qp.apply(op)
            if i == error_idx:
                error_wire = rng.choice(list(op.wires)) # Insert coherent error on one of the gates' wires
                qp.PauliX(wires=error_wire)
    
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()
    
    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def ideal_circuit(index):
        qp.BasisState(index, wires=control_wires)
        qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()

    r = qp.specs(ideal_circuit)(0)["resources"]

    num_qubits = len(control_wires) + len(target_wires) + len(work_wires)
    oneQ = r.gate_sizes.get(1,0)
    twoQ = r.gate_sizes.get(2,0)

    if print_out:
        print("Number of qubits: ", num_qubits)
        print("One-qubit gates: ", oneQ)
        print("Two-qubit gates: ", twoQ)
        print(f"Depth: {r.depth}")
        
    ideal_fidelities = []
    noisy_fidelities = []
    state_fidelities = []
    full_start_time = time.perf_counter()

    for index in range(2**num_index_bits):
        start_time = time.perf_counter()
        
        # Expected |address>|data> state
        probs_length = list(map(int, f"{index:0{num_index_bits}b}")) + list(map(int, bitstrings[index]))
        probs_index = int("".join(map(str, probs_length)), 2)

        # Ideal Circuit Probability of retrieving the correct address + data.
        ideal_probs, ideal_state = ideal_circuit(index)
        ideal_fidelity = ideal_probs[probs_index]
        ideal_fidelity = np.clip(np.real(ideal_fidelity), 0.0, 1.0)

        # Noisy Circuit Probability of retrieving the correct address + data.
        noisy_probs, noisy_state = noisy_circuit(index)
        noisy_fidelity = noisy_probs[probs_index]
        noisy_fidelity = np.clip(np.real(noisy_fidelity), 0.0, 1.0)

        # State Fidelity
        state_fidelity = np.abs(np.vdot(ideal_state, noisy_state))**2

        elapsed = time.perf_counter() - start_time
        
        if print_out: 
            print(f"Index {index} took {elapsed:.6f} s")
            print(f"Ideal QROM Retrieval Fidelity: {ideal_fidelity}")
            print(f"Noisy QROM Retrieval Fidelity: {noisy_fidelity}")
            print(f"State Fidelity: {state_fidelity}\n")
        
        ideal_fidelities.append(float(ideal_fidelity))
        noisy_fidelities.append(float(noisy_fidelity))
        state_fidelities.append(float(state_fidelity))
        
    full_elapsed = time.perf_counter() - full_start_time    
    
    if print_out: 
        print(f"Total Time: {full_elapsed:.6f} s")
        print(f"Final Ideal Retrieval Fidelity is {np.mean(ideal_fidelities)}")
        print(f"Final Noisy Retrieval Fidelity is {np.mean(noisy_fidelities)}")
        print(f"Final State Fidelity is {np.mean(state_fidelities)}")

Interesting pattern: the more swap bits we add, the more noise resistant we are in the retrieval fidelity. For 0 swap bits, the retrieval fidelity is bad for all of the indices. For 1 swap bit, the retrieval is bad for about half of the indices. For 2 swap bits, the retrieval is bad for about a quarter of the indices.

In [52]:
num_index_bits = 3
num_swap_bits = 0
bits_per_value = 3

bitstrings = generate_bitstrings(2**num_index_bits, bits_per_value, worstcase=True)
evaluate_QROM(bitstrings, num_index_bits, num_swap_bits, bits_per_value, print_out=True)


Number of qubits:  6
One-qubit gates:  1158
Two-qubit gates:  576
Depth: 1071
Index 0 took 2.053342 s
Ideal QROM Retrieval Fidelity: 0.9999999999999705
Noisy QROM Retrieval Fidelity: 3.0694204247194114e-28
State Fidelity: 1.2887358770350162e-28

Index 1 took 1.997136 s
Ideal QROM Retrieval Fidelity: 0.9999999999999689
Noisy QROM Retrieval Fidelity: 3.594195530761646e-28
State Fidelity: 1.7538656243645892e-28

Index 2 took 2.129286 s
Ideal QROM Retrieval Fidelity: 0.9999999999999734
Noisy QROM Retrieval Fidelity: 3.279553354227167e-30
State Fidelity: 5.892809922527913e-30

Index 3 took 1.972000 s
Ideal QROM Retrieval Fidelity: 0.999999999999974
Noisy QROM Retrieval Fidelity: 3.035523960346979e-28
State Fidelity: 1.4517550747657374e-28

Index 4 took 1.974940 s
Ideal QROM Retrieval Fidelity: 0.9999999999999727
Noisy QROM Retrieval Fidelity: 9.598462609035116e-31
State Fidelity: 4.536773342551621e-31

Index 5 took 1.971677 s
Ideal QROM Retrieval Fidelity: 0.9999999999999749
Noisy QROM Retr

In [51]:
num_index_bits = 3
num_swap_bits = 1
bits_per_value = 3

bitstrings = generate_bitstrings(2**num_index_bits, bits_per_value, worstcase=True)
evaluate_QROM(bitstrings, num_index_bits, num_swap_bits, bits_per_value, print_out=True)


Number of qubits:  9
One-qubit gates:  337
Two-qubit gates:  168
Depth: 272
Index 0 took 0.432799 s
Ideal QROM Retrieval Fidelity: 0.9999999999999958
Noisy QROM Retrieval Fidelity: 0.999999999999987
State Fidelity: 2.279852993774183e-30

Index 1 took 0.440267 s
Ideal QROM Retrieval Fidelity: 0.9999999999999937
Noisy QROM Retrieval Fidelity: 6.218438237967194e-30
State Fidelity: 2.652768492746294e-30

Index 2 took 0.573966 s
Ideal QROM Retrieval Fidelity: 0.9999999999999947
Noisy QROM Retrieval Fidelity: 3.74939945665458e-31
State Fidelity: 1.8372057337607346e-31

Index 3 took 0.441862 s
Ideal QROM Retrieval Fidelity: 0.9999999999999944
Noisy QROM Retrieval Fidelity: 5.278634939559695e-30
State Fidelity: 2.2704638966310997e-30

Index 4 took 0.441177 s
Ideal QROM Retrieval Fidelity: 0.9999999999999938
Noisy QROM Retrieval Fidelity: 3.7493994566545865e-33
State Fidelity: 3.749399456654564e-33

Index 5 took 0.435899 s
Ideal QROM Retrieval Fidelity: 0.9999999999999954
Noisy QROM Retrieval F

In [53]:
num_index_bits = 3
num_swap_bits = 2
bits_per_value = 3

bitstrings = generate_bitstrings(2**num_index_bits, bits_per_value, worstcase=True)
evaluate_QROM(bitstrings, num_index_bits, num_swap_bits, bits_per_value, print_out=True)


Number of qubits:  15
One-qubit gates:  119
Two-qubit gates:  96
Depth: 93
Index 0 took 0.277290 s
Ideal QROM Retrieval Fidelity: 0.9999999999999972
Noisy QROM Retrieval Fidelity: 1.2086111726653078e-30
State Fidelity: 3.749399456654609e-33

Index 1 took 0.257651 s
Ideal QROM Retrieval Fidelity: 0.999999999999997
Noisy QROM Retrieval Fidelity: 0.9999999999999929
State Fidelity: 3.749399456654605e-33

Index 2 took 0.251916 s
Ideal QROM Retrieval Fidelity: 0.9999999999999973
Noisy QROM Retrieval Fidelity: 3.6428348504594466e-31
State Fidelity: 3.749399456654608e-33

Index 3 took 0.288841 s
Ideal QROM Retrieval Fidelity: 0.9999999999999971
Noisy QROM Retrieval Fidelity: 0.9999999999999944
State Fidelity: 3.749399456654619e-33

Index 4 took 0.274128 s
Ideal QROM Retrieval Fidelity: 0.9999999999999973
Noisy QROM Retrieval Fidelity: 0.9999999999999932
State Fidelity: 3.749399456654608e-33

Index 5 took 0.278444 s
Ideal QROM Retrieval Fidelity: 0.9999999999999971
Noisy QROM Retrieval Fidelity

### Backtracking but for specific error_idx

In [81]:
def list_error_locations(bitstrings, num_index_bits, num_swap_bits=0, bits_per_value=2, print_out=False, error_idx=0):

    if(len(bitstrings) != 2**num_index_bits or len(bitstrings[0]) != bits_per_value):
        print(f"Error: Mismatch in size of bitstrings and expected number of num_index_bits and bits_per_value. ({len(bitstrings)}, {len(bitstrings[0])}) != ({2**num_index_bits}, {bits_per_value})")
        return -1, -1, -1, -1, -1
    
    control_wires = np.arange(num_index_bits)
    target_wires = np.arange(num_index_bits, num_index_bits + bits_per_value)
    work_wires = np.arange(target_wires[-1] + 1, target_wires[-1] + 1 + bits_per_value*(2**num_swap_bits - 1))

    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def noisy_circuit(index, epsilon=0.05, seed=None):
    
        rng = np.random.default_rng(seed)
        qp.BasisState(index, wires=control_wires)
        qrom = qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
    
        ops = qrom.decomposition()         # Decompose into primitive operations
        # error_idx = 0
        # error_idx = rng.integers(len(ops)) # Choose a random location inside the QROM

        for idx, op in enumerate(ops):
            print(f"{idx:3}: {op.name:20} wires={list(op.wires)}")

        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()

    noisy_circuit(0) # All indices have the same.

In [69]:
def evaluate_QROM(bitstrings, num_index_bits, num_swap_bits=0, bits_per_value=2, print_out=False, error_idx=0):

    if(len(bitstrings) != 2**num_index_bits or len(bitstrings[0]) != bits_per_value):
        print(f"Error: Mismatch in size of bitstrings and expected number of num_index_bits and bits_per_value. ({len(bitstrings)}, {len(bitstrings[0])}) != ({2**num_index_bits}, {bits_per_value})")
        return -1, -1, -1, -1, -1
    
    control_wires = np.arange(num_index_bits)
    target_wires = np.arange(num_index_bits, num_index_bits + bits_per_value)
    work_wires = np.arange(target_wires[-1] + 1, target_wires[-1] + 1 + bits_per_value*(2**num_swap_bits - 1))

    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def noisy_circuit(index, epsilon=0.05, seed=None):
    
        rng = np.random.default_rng(seed)
        qp.BasisState(index, wires=control_wires)
        qrom = qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
    
        ops = qrom.decomposition()         # Decompose into primitive operations
        # error_idx = 0
        # error_idx = rng.integers(len(ops)) # Choose a random location inside the QROM

        error_locations = []

        for idx, op in enumerate(ops):
            for wire in op.wires:
                error_locations.append((idx, wire))
        return error_locations
        
        for i, op in enumerate(ops):
            qp.apply(op)
            if i == error_idx:
                error_wire = rng.choice(list(op.wires)) # Insert coherent error on one of the gates' wires
                qp.PauliX(wires=error_wire)
    
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()
    
    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def ideal_circuit(index):
        qp.BasisState(index, wires=control_wires)
        qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()

    r = qp.specs(ideal_circuit)(0)["resources"]

    num_qubits = len(control_wires) + len(target_wires) + len(work_wires)
    oneQ = r.gate_sizes.get(1,0)
    twoQ = r.gate_sizes.get(2,0)

    if print_out:
        print("Number of qubits: ", num_qubits)
        print("One-qubit gates: ", oneQ)
        print("Two-qubit gates: ", twoQ)
        print(f"Depth: {r.depth}")
        
    ideal_fidelities = []
    noisy_fidelities = []
    state_fidelities = []
    full_start_time = time.perf_counter()

    for index in range(2**num_index_bits):
        start_time = time.perf_counter()
        
        # Expected |address>|data> state
        probs_length = list(map(int, f"{index:0{num_index_bits}b}")) + list(map(int, bitstrings[index]))
        probs_index = int("".join(map(str, probs_length)), 2)

        # Ideal Circuit Probability of retrieving the correct address + data.
        ideal_probs, ideal_state = ideal_circuit(index)
        ideal_fidelity = ideal_probs[probs_index]
        ideal_fidelity = np.clip(np.real(ideal_fidelity), 0.0, 1.0)

        # Noisy Circuit Probability of retrieving the correct address + data.
        noisy_probs, noisy_state = noisy_circuit(index)
        noisy_fidelity = noisy_probs[probs_index]
        noisy_fidelity = np.clip(np.real(noisy_fidelity), 0.0, 1.0)

        # State Fidelity
        state_fidelity = np.abs(np.vdot(ideal_state, noisy_state))**2

        elapsed = time.perf_counter() - start_time
        
        if print_out and noisy_fidelity < 0.1: 
            print(f"Index {index} took {elapsed:.6f} s")
            print(f"Noisy QROM Retrieval Fidelity: {noisy_fidelity}")
        
        ideal_fidelities.append(float(ideal_fidelity))
        noisy_fidelities.append(float(noisy_fidelity))
        state_fidelities.append(float(state_fidelity))

        
        
    full_elapsed = time.perf_counter() - full_start_time    
    
    if print_out: 
        print(f"Total Time: {full_elapsed:.6f} s")
        print(f"Final Noisy Retrieval Fidelity is {np.mean(noisy_fidelities)}\n")

In [71]:
num_index_bits = 4
num_swap_bits = 2
bits_per_value = 3

for i in range(4):
    bitstrings = generate_bitstrings(2**num_index_bits, bits_per_value, worstcase=True)
    evaluate_QROM(bitstrings, num_index_bits, num_swap_bits, bits_per_value, print_out=True, error_idx=i)


Number of qubits:  16
One-qubit gates:  706
Two-qubit gates:  360
Depth: 539
Index 0 took 1.977100 s
Noisy QROM Retrieval Fidelity: 4.410326301251142e-30
Index 1 took 1.993018 s
Noisy QROM Retrieval Fidelity: 6.542784142650517e-30
Index 3 took 2.095096 s
Noisy QROM Retrieval Fidelity: 7.394235331387185e-30
Index 5 took 2.051365 s
Noisy QROM Retrieval Fidelity: 4.536773342551991e-31
Index 11 took 2.140927 s
Noisy QROM Retrieval Fidelity: 6.43523226695286e-30
Index 13 took 2.264279 s
Noisy QROM Retrieval Fidelity: 4.536773342551985e-31
Index 14 took 2.000504 s
Noisy QROM Retrieval Fidelity: 8.564055150082058e-30
Total Time: 32.828040 s
Final Noisy Retrieval Fidelity is 0.5624999999999829

Number of qubits:  16
One-qubit gates:  706
Two-qubit gates:  360
Depth: 539
Index 5 took 2.541136 s
Noisy QROM Retrieval Fidelity: 3.749399456654521e-33
Index 7 took 2.043814 s
Noisy QROM Retrieval Fidelity: 7.577331283965431e-30
Index 9 took 2.107526 s
Noisy QROM Retrieval Fidelity: 4.329229507989145e

In [85]:
num_index_bits = 4
num_swap_bits = 2
bits_per_value = 3
bitstrings = generate_bitstrings(2**num_index_bits, bits_per_value, worstcase=True)
list_error_locations(bitstrings, num_index_bits, num_swap_bits, bits_per_value, print_out=True, error_idx=i)

  0: Select               wires=[np.int64(0), np.int64(1), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]
  1: C(Prod)              wires=[np.int64(2), np.int64(9), np.int64(15), np.int64(8), np.int64(14), np.int64(7), np.int64(13)]
  2: C(Prod)              wires=[np.int64(2), np.int64(6), np.int64(12), np.int64(5), np.int64(11), np.int64(4), np.int64(10)]
  3: C(Prod)              wires=[np.int64(3), np.int64(6), np.int64(9), np.int64(5), np.int64(8), np.int64(4), np.int64(7)]


Next Steps: Find out all the failing locations for every combination of error_idx and error_wire. In your current implementation, the error is injected after the chosen operation, not before it.

### Adding the StronglyEntanglingLayers randomly to see if it helps

In [42]:
def evaluate_QROM(bitstrings, num_index_bits, num_swap_bits=0, bits_per_value=2, print_out=False):

    if(len(bitstrings) != 2**num_index_bits or len(bitstrings[0]) != bits_per_value):
        print(f"Error: Mismatch in size of bitstrings and expected number of num_index_bits and bits_per_value. ({len(bitstrings)}, {len(bitstrings[0])}) != ({2**num_index_bits}, {bits_per_value})")
        return -1, -1, -1, -1, -1
    
    control_wires = np.arange(num_index_bits)
    target_wires = np.arange(num_index_bits, num_index_bits + bits_per_value)
    work_wires = np.arange(target_wires[-1] + 1, target_wires[-1] + 1 + bits_per_value*(2**num_swap_bits - 1))

    def recovery_layer(weights, wires):
        qp.StronglyEntanglingLayers(weights, wires=wires)
    
    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def noisy_circuit(index, weights, seed=None):
    
        rng = np.random.default_rng(seed)
        qp.BasisState(index, wires=control_wires)
        qrom = qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
        ops = qrom.decomposition()
    
        # error_idx = rng.integers(len(ops))
        
        for i, op in enumerate(ops):
            qp.apply(op)
            if i == error_idx:
                error_wire = rng.choice(list(op.wires))
                qp.PauliX(wires=error_wire)
    
        recovery_layer(weights, wires=range(num_qubits))
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()
    
    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def ideal_circuit(index):
        qp.BasisState(index, wires=control_wires)
        qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()

    r = qp.specs(ideal_circuit)(0)["resources"]

    num_qubits = len(control_wires) + len(target_wires) + len(work_wires)
    oneQ = r.gate_sizes.get(1,0)
    twoQ = r.gate_sizes.get(2,0)

    if print_out:
        print("Number of qubits: ", num_qubits)
        print("One-qubit gates: ", oneQ)
        print("Two-qubit gates: ", twoQ)
        print(f"Depth: {r.depth}")
        
    ideal_fidelities = []
    noisy_fidelities = []
    state_fidelities = []
    full_start_time = time.perf_counter()

    for index in range(2**num_index_bits):
        start_time = time.perf_counter()
        
        # Expected |address>|data> state
        probs_length = list(map(int, f"{index:0{num_index_bits}b}")) + list(map(int, bitstrings[index]))
        probs_index = int("".join(map(str, probs_length)), 2)

        # Ideal Circuit Probability of retrieving the correct address + data.
        ideal_probs, ideal_state = ideal_circuit(index)
        ideal_fidelity = ideal_probs[probs_index]
        ideal_fidelity = np.clip(np.real(ideal_fidelity), 0.0, 1.0)

        # Noisy Circuit Probability of retrieving the correct address + data.
        weights = np.full((1, num_qubits, 3), np.pi/100)
        noisy_probs, noisy_state = noisy_circuit(index, weights)
        noisy_fidelity = noisy_probs[probs_index]
        noisy_fidelity = np.clip(np.real(noisy_fidelity), 0.0, 1.0)

        # State Fidelity
        state_fidelity = np.abs(np.vdot(ideal_state, noisy_state))**2

        elapsed = time.perf_counter() - start_time
        
        if print_out: 
            print(f"Index {index} took {elapsed:.6f} s")
            print(f"Ideal QROM Retrieval Fidelity: {ideal_fidelity}")
            print(f"Noisy QROM Retrieval Fidelity: {noisy_fidelity}")
            print(f"State Fidelity: {state_fidelity}\n")
        
        ideal_fidelities.append(float(ideal_fidelity))
        noisy_fidelities.append(float(noisy_fidelity))
        state_fidelities.append(float(state_fidelity))
        
    full_elapsed = time.perf_counter() - full_start_time    
    
    if print_out: 
        print(f"Total Time: {full_elapsed:.6f} s")
        print(f"Final Ideal Retrieval Fidelity is {np.mean(ideal_fidelities)}")
        print(f"Final Noisy Retrieval Fidelity is {np.mean(noisy_fidelities)}")
        print(f"Final State Fidelity is {np.mean(state_fidelities)}")

In [43]:
num_index_bits = 4
num_swap_bits = 2
bits_per_value = 4

bitstrings = generate_bitstrings(2**num_index_bits, bits_per_value, worstcase=True)
evaluate_QROM(bitstrings, num_index_bits, num_swap_bits, bits_per_value, print_out=True)


Number of qubits:  20
One-qubit gates:  939
Two-qubit gates:  480
Depth: 708
Index 0 took 23.593927 s
Ideal QROM Retrieval Fidelity: 0.9999999999999845
Noisy QROM Retrieval Fidelity: 1.4955195354842073e-11
State Fidelity: 8.074479705218535e-42

Index 1 took 23.038114 s
Ideal QROM Retrieval Fidelity: 0.999999999999984
Noisy QROM Retrieval Fidelity: 4.428862565633029e-14
State Fidelity: 3.2719250913084814e-38

Index 2 took 22.609562 s
Ideal QROM Retrieval Fidelity: 0.9999999999999831
Noisy QROM Retrieval Fidelity: 3.690653620540235e-15
State Fidelity: 1.992625776595355e-45



KeyboardInterrupt: 

# Recovering a Bit Flip

In [ ]:
def evaluate_QROM(bitstrings, num_index_bits, num_swap_bits=0, bits_per_value=2, print_out=False):

    if(len(bitstrings) != 2**num_index_bits or len(bitstrings[0]) != bits_per_value):
        print(f"Error: Mismatch in size of bitstrings and expected number of num_index_bits and bits_per_value. ({len(bitstrings)}, {len(bitstrings[0])}) != ({2**num_index_bits}, {bits_per_value})")
        return -1, -1, -1, -1, -1
    
    control_wires = np.arange(num_index_bits)
    target_wires = np.arange(num_index_bits, num_index_bits + bits_per_value)
    work_wires = np.arange(target_wires[-1] + 1, target_wires[-1] + 1 + bits_per_value*(2**num_swap_bits - 1))

    # Circuit Implementation ###################
    
    def recovery_layer(weights, wires):
        qp.StronglyEntanglingLayers(weights, wires=wires)
    
    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def ideal_circuit(index):
        qp.BasisState(index, wires=control_wires)
        qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()

    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def noisy_circuit(index, epsilon=0.05, seed=None):
    
        rng = np.random.default_rng(seed)
        qp.BasisState(index, wires=control_wires)
        qrom = qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
    
        ops = qrom.decomposition()         # Decompose into primitive operations
        error_idx = rng.integers(len(ops)) # Choose a random location inside the QROM
    
        for i, op in enumerate(ops):
            qp.apply(op)
            if i == error_idx:
                error_wire = rng.choice(list(op.wires)) # Insert coherent error on one of the gates' wires
                qp.PauliX(wires=error_wire)
    
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()

    @partial(qp.compile, basis_set="CNOT")
    @qp.qnode(qp.device("default.qubit"))
    def recovered_circuit(index, weights, seed=None):
    
        rng = np.random.default_rng(seed)
        qp.BasisState(index, wires=control_wires)
        qrom = qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
        ops = qrom.decomposition()
    
        error_idx = rng.integers(len(ops))
        for i, op in enumerate(ops):
            qp.apply(op)
            if i == error_idx:
                error_wire = rng.choice(list(op.wires))
                qp.PauliX(wires=error_wire)
    
        recovery_layer(weights, wires=range(num_qubits))
        return qp.probs(wires=list(control_wires) + list(target_wires)), qp.state()

    # State Fidelity #############################
    def state_fidelity(ideal_fn, noisy_fn, num_index_bits, num_seeds=5):
        
        total = 0.0
        
        for index in range(2**num_index_bits):
            _, psi_ideal = ideal_fn(index)
            
            for seed in range(num_seeds):
                _, psi_noisy = noisy_fn(index, seed=seed)
                total += np.abs(np.vdot(psi_ideal, psi_noisy))**2
    
        return total / (2**num_index_bits * num_seeds)

    # Get Training Examples #########################
    
    training_examples = []
    for index in range(2**num_index_bits):
        
        index_time = time.perf_counter()
        psi_ideal = ideal_circuit(index)
        print(f"Ideal Circuit for Index {index} took {time.perf_counter() - index_time:.6f} s")
        
        for seed in range(5):
            noisy_time = time.perf_counter()
            psi_noisy = recovered_circuit(index=index, weights=np.zeros((1,num_qubits,3)),seed=seed)
            training_examples.append((psi_ideal, psi_noisy))
            print(f"\tNoisy Circuit for Index {index}, Seed {seed} took {time.perf_counter() - noisy_time:.6f} s")
    with open("training_examples.pkl", "wb") as f:
        pickle.dump(training_examples, f)

    # Gradient Descent Model
    weights = np.random.normal(0, 0.01, (1, num_qubits, 3), requires_grad=True)
    def cost(weights):
        return np.mean([
            1 - np.abs(np.vdot(ideal_circuit(index), recovered_circuit(index, weights, seed))**2)
            for index in range(2**num_index_bits) for seed in range(5)
        ])

    opt = qml.AdamOptimizer(stepsize=0.05)
    for step in range(100):
        weights = opt.step(cost, weights)
        if step % 10 == 0:
            c = cost(weights)
            print(f"Step {step}: Loss={c:.6f}")
            

In [ ]:
# Getting training samples of randomly inserted errors
num_index_bits = 4
num_swap_bits = 2
bits_per_value = 4

bitstrings = generate_bitstrings(2**num_index_bits, bits_per_value, worstcase=True)
evaluate_QROM(bitstrings, num_index_bits, num_swap_bits, bits_per_value, print_out=True)